# 초능력자 봇 A100 대량 파라미터 탐색 v3.2.1 FIXED

- 2-1 Categorical segment 오류 수정
- #5 유저 텐서 변수명 호환성 보강
- #9 stage1_res 복구 로직 추가
- #10 stage2_res 복구 로직 추가
- 대량 탐색 전용: 실제 실행 전 QUICK_MODE/배치 크기 확인 권장

In [1]:
# ================================================================
# 0. 환경 설정
# ================================================================
import os, re, math, json, random, warnings, gc
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd

import torch

warnings.filterwarnings("ignore")

try:
    from google.colab import files
    IN_COLAB = True
except Exception:
    IN_COLAB = False

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", DEVICE)
if DEVICE == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

OUT_DIR = Path("/content/chaoneung_v32_outputs") if IN_COLAB else Path("./chaoneung_v32_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if DEVICE == "cuda":
    torch.cuda.manual_seed_all(SEED)

DEVICE: cuda
GPU: NVIDIA A100-SXM4-40GB


In [2]:
# ================================================================
# 1. 대량 탐색 설정
# ================================================================
# 빠른 테스트: True
# A100 본실험: False
QUICK_MODE = False

# 1차 대량 탐색
if QUICK_MODE:
    STAGE1_N_PARAM = 4096
    STAGE1_R_MC = 2
    STAGE1_BATCH_SIZE = 256
    STAGE2_N_PARAM = 4096
    STAGE2_R_MC = 2
    STAGE2_BATCH_SIZE = 256
    VALIDATE_TOP_N = 50
    VALIDATE_R_MC = 8
else:
    STAGE1_N_PARAM = 65536
    STAGE1_R_MC = 4
    STAGE1_BATCH_SIZE = 1024

    # 2차는 elite 주변 재탐색
    STAGE2_N_PARAM = 131072
    STAGE2_R_MC = 4
    STAGE2_BATCH_SIZE = 1024

    # 최종 상위 후보 정밀검증
    VALIDATE_TOP_N = 500
    VALIDATE_R_MC = 64

T_DAYS = 180
SIM_MAX_ENHANCE_TRIES_PER_DAY = 3
FUSE_INTERVAL_DAYS = 7
SAVE_EVERY_BATCH = True
KEEP_TOP_K = 5000

# 제약 목표
TARGET_D20_MIN = 250_000
TARGET_D20_MAX = 2_000_000
TARGET_LATE_SHARE_MAX = 0.70

TARGET_REACH_D5 = 0.40
TARGET_REACH_D10 = 0.12
MAX_REACH_D15 = 0.03
MAX_REACH_D20 = 0.001

MAX_HAS_A = 0.02
MAX_HAS_S = 0.001

TARGET_SINK = 0.75
MIN_SINK = 0.65
MAX_SINK = 0.90
MAX_P99_P50 = 7.0

ENFORCE_MONOTONIC_SUCCESS = True

print({
    "QUICK_MODE": QUICK_MODE,
    "STAGE1_N_PARAM": STAGE1_N_PARAM,
    "STAGE2_N_PARAM": STAGE2_N_PARAM,
    "T_DAYS": T_DAYS,
    "DEVICE": DEVICE
})

{'QUICK_MODE': False, 'STAGE1_N_PARAM': 65536, 'STAGE2_N_PARAM': 131072, 'T_DAYS': 180, 'DEVICE': 'cuda'}


In [3]:
# ================================================================
# 2. 파일 로드 및 유저 파싱
# ================================================================

def find_or_upload_txt():
    candidates = [
        "/content/초능력자봇_전체데이터_ML용.txt",
        "/mnt/data/초능력자봇_전체데이터_ML용.txt",
        "./초능력자봇_전체데이터_ML용.txt",
    ]
    for p in candidates:
        if Path(p).exists():
            return Path(p)

    if IN_COLAB:
        print("TXT 파일을 업로드하세요: 초능력자봇_전체데이터_ML용.txt")
        uploaded = files.upload()
        for name in uploaded.keys():
            if name.endswith(".txt"):
                return Path("/content") / name

    raise FileNotFoundError("초능력자봇_전체데이터_ML용.txt 파일을 찾지 못했습니다.")

TXT_PATH = find_or_upload_txt()
print("TXT_PATH:", TXT_PATH)

raw_text = TXT_PATH.read_text(encoding="utf-8", errors="ignore")
print("chars:", len(raw_text))


def parse_production_users(text):
    """
    production 전체 유저 랭킹 테이블에서
    nickname, score, correct, total, accuracy, last_attendance를 파싱.
    """
    rows = []
    in_table = False
    for line in text.splitlines():
        line = line.strip()
        if line.startswith("nickname,score(P),correct,total,accuracy(%),last_attendance"):
            in_table = True
            continue
        if not in_table:
            continue
        if not line:
            continue
        # 다음 섹션에 들어가면 종료
        if line.startswith("─") or line.startswith("━") or line.startswith("[") or line.startswith("※"):
            if rows:
                break
            continue

        parts = [p.strip() for p in line.split(",")]
        if len(parts) < 6:
            continue
        nick = parts[0]
        try:
            score = int(float(parts[1]))
            correct = int(float(parts[2]))
            total = int(float(parts[3]))
            accuracy = float(parts[4])
            last_attendance = parts[5]
        except Exception:
            continue

        rows.append({
            "nickname": nick,
            "score": score,
            "correct": correct,
            "total": total,
            "accuracy": accuracy,
            "last_attendance": last_attendance
        })

    df = pd.DataFrame(rows)
    if len(df) == 0:
        raise ValueError("유저 테이블을 파싱하지 못했습니다.")
    return df

users = parse_production_users(raw_text)
print("parsed users:", len(users))
display(users.head())

TXT 파일을 업로드하세요: 초능력자봇_전체데이터_ML용.txt


Saving 초능력자봇_전체데이터_ML용.txt to 초능력자봇_전체데이터_ML용.txt
TXT_PATH: /content/초능력자봇_전체데이터_ML용.txt
chars: 210896
parsed users: 4690


,nickname,score,correct,total,accuracy,last_attendance
0,능력자_ea4997,160824,3161,3230,97.9,2026-05-01
1,능력자_d5dcad,75403,988,1688,58.5,2026-05-03
2,능력자_66a071,61145,1165,1288,90.5,2026-05-04
3,부운영진_6a1c4c,53397,948,1039,91.2,2026-05-04
4,능력자_fb3f51,43235,575,1342,42.8,없음


In [4]:
# ================================================================
# 2-1. 요약값 파싱 및 부족 유저 synthetic tail 보정
# ================================================================

import re
import numpy as np
import pandas as pd


def rex(pattern, text, default=None, cast=float):
    m = re.search(pattern, text)
    if not m:
        return default
    try:
        return cast(m.group(1).replace(",", ""))
    except Exception:
        return default

# ------------------------------------------------
# 1) 보고서에 적힌 전체 집계값 파싱
# ------------------------------------------------
total_users_reported = rex(r"total_users:\s*([0-9,]+)", raw_text, default=len(users), cast=int)
active_users_reported = rex(r"active_users\s*\(total>0\):\s*([0-9,]+)", raw_text, default=None, cast=int)
total_correct_reported = rex(r"total_correct:\s*([0-9,]+)", raw_text, default=None, cast=int)
total_played_reported = rex(r"total_played:\s*([0-9,]+)", raw_text, default=None, cast=int)

print("reported:", total_users_reported, active_users_reported, total_correct_reported, total_played_reported)

# ------------------------------------------------
# 2) 부족 유저 synthetic tail 보정
# ------------------------------------------------
# TXT가 길어서 일부 유저만 파싱된 경우, 보고서의 score 구간 분포를 이용해
# 하위 유저를 synthetic tail로 채운다. 상위권은 실제 parsed 유저를 그대로 사용한다.

score_bins = [
    ("0-100P", 0, 100, 3706),
    ("100-500P", 100, 500, 1851),
    ("500-1kP", 500, 1000, 603),
    ("1k-5kP", 1000, 5000, 620),
    ("5k-10kP", 5000, 10000, 97),
    ("10k-50kP", 10000, 50000, 64),
    ("50k-100kP", 50000, 100000, 3),
    ("100k+P", 100000, 170000, 1),
]


def add_synthetic_tail(df, target_n):
    df = df.copy()
    if len(df) >= target_n:
        print("synthetic tail 생성 없음")
        return df

    need = int(target_n - len(df))
    print(f"synthetic tail 생성: {need}명")

    rng = np.random.default_rng(SEED)
    synth = []

    cut_bins = [-1, 100, 500, 1000, 5000, 10000, 50000, 100000, 10**9]
    cut_labels = [b[0] for b in score_bins]
    current_counts = pd.cut(df["score"], bins=cut_bins, labels=cut_labels).value_counts().to_dict()

    idx = 0
    for label, lo, hi, target_count in score_bins:
        cur = int(current_counts.get(label, 0))
        n_add = max(0, int(target_count) - cur)

        for _ in range(n_add):
            if len(synth) >= need:
                break

            if lo == 0:
                score = int(rng.integers(lo, hi + 1))
            elif hi > 5000:
                score = int(np.exp(rng.uniform(np.log(lo + 1), np.log(hi + 1))) - 1)
            else:
                score = int(rng.uniform(lo, hi + 1))

            # score와 느슨하게 연결된 누적 total/correct 생성
            total = int(rng.poisson(max(1.0, score / 60.0)))
            correct = int(rng.binomial(max(total, 1), 0.3359)) if total > 0 else 0
            acc = 100.0 * correct / max(total, 1)

            synth.append({
                "nickname": f"synthetic_{idx:05d}",
                "score": score,
                "correct": correct,
                "total": total,
                "accuracy": acc,
                "last_attendance": "없음",
                "source": "synthetic_tail",
            })
            idx += 1
        if len(synth) >= need:
            break

    while len(synth) < need:
        synth.append({
            "nickname": f"synthetic_{idx:05d}",
            "score": int(rng.integers(0, 101)),
            "correct": 0,
            "total": 0,
            "accuracy": 0.0,
            "last_attendance": "없음",
            "source": "synthetic_tail",
        })
        idx += 1

    df["source"] = df.get("source", "parsed")
    return pd.concat([df, pd.DataFrame(synth)], ignore_index=True)


users = add_synthetic_tail(users, int(total_users_reported))

# ------------------------------------------------
# 3) 타입/파생변수 안정화
# ------------------------------------------------
for col in ["score", "correct", "total", "accuracy"]:
    users[col] = pd.to_numeric(users[col], errors="coerce").fillna(0.0).astype(float)

users["accuracy_frac"] = np.where(
    users["total"] > 0,
    users["correct"] / np.maximum(users["total"], 1.0),
    0.0,
)
users["accuracy_frac"] = np.clip(users["accuracy_frac"], 0.0, 1.0)
users["accuracy"] = users["accuracy_frac"] * 100.0
users["is_active"] = users["total"] > 0
users["recent_active"] = users["last_attendance"].astype(str).str.contains(
    r"2026-05-0[3-5]|2026-05-04|2026-05-05", regex=True
).astype(float)

users["log_score"] = np.log1p(users["score"])
users["log_correct"] = np.log1p(users["correct"])
users["log_total"] = np.log1p(users["total"])

# ------------------------------------------------
# 4) 세그먼트 분류
# ------------------------------------------------
q_score = users["score"].rank(pct=True, method="average")
q_correct = users["correct"].rank(pct=True, method="average")
activity_score = 0.55 * q_correct + 0.25 * q_score + 0.20 * users["recent_active"]
users["activity_score"] = activity_score

users["segment"] = pd.cut(
    activity_score,
    bins=[-0.001, 0.40, 0.75, 0.95, 0.99, 1.001],
    labels=["low", "mid", "high", "whale", "ultra"],
)

# pd.cut은 Categorical을 반환하므로 새 값("inactive")을 넣기 전에 object로 변환해야 한다.
users["segment"] = users["segment"].astype("object")
users.loc[~users["is_active"], "segment"] = "inactive"
users["segment"] = users["segment"].fillna("inactive").astype(str)

users = users.sort_values(["score", "correct"], ascending=False).reset_index(drop=True)

display(users["segment"].value_counts(dropna=False))
display(users.head(10))
users.to_csv(OUT_DIR / "parsed_users_segments_v32.csv", index=False, encoding="utf-8-sig")


reported: 6945 4690 57073 169928
synthetic tail 생성: 2255명


,count
segment,
low,2805
mid,2766
inactive,692
high,489
whale,135
ultra,58


,nickname,score,correct,total,accuracy,last_attendance,source,accuracy_frac,is_active,recent_active,log_score,log_correct,log_total,activity_score,segment
0,능력자_ea4997,160824.0,3161.0,3230.0,97.863777,2026-05-01,parsed,0.978638,True,0.0,11.988072,8.058960,8.080547,0.800000,high
1,능력자_d5dcad,75403.0,988.0,1688.0,58.530806,2026-05-03,parsed,0.585308,True,1.0,11.230616,6.896694,7.431892,0.999806,ultra
2,능력자_66a071,61145.0,1165.0,1288.0,90.450311,2026-05-04,parsed,0.904503,True,1.0,11.021020,7.061334,7.161622,0.999849,ultra
3,부운영진_6a1c4c,53397.0,948.0,1039.0,91.241578,2026-05-04,parsed,0.912416,True,1.0,10.885529,6.855409,6.946976,0.999654,ultra
4,능력자_fb3f51,43235.0,575.0,1342.0,42.846498,없음,parsed,0.428465,True,0.0,10.674429,6.356108,7.202661,0.799460,high
5,능력자_97b77b,36339.0,714.0,807.0,88.475836,2026-05-04,parsed,0.884758,True,1.0,10.500674,6.572283,6.694562,0.999503,ultra
6,능력자_609a16,33676.0,439.0,702.0,62.535613,2026-05-03,parsed,0.625356,True,1.0,10.424570,6.086775,6.555357,0.999150,ultra
7,능력자_bb5302,28442.0,454.0,591.0,76.818951,2026-05-05,parsed,0.768190,True,1.0,10.255657,6.120297,6.383507,0.999194,ultra
8,능력자_4ae102,28300.0,397.0,878.0,45.216401,2026-05-04,parsed,0.452164,True,1.0,10.250652,5.986452,6.778785,0.998999,ultra
9,능력자_42d98a,24913.0,471.0,607.0,77.594728,2026-05-04,parsed,0.775947,True,1.0,10.123185,6.156979,6.410175,0.999201,ultra


In [5]:
# ================================================================
# 3. 기본 파라미터 및 탐색 범위
# ================================================================
BASE = dict(
    base_effect=np.array([0.022,0.044,0.087,0.164,0.273],float),
    level_bonus=0.05, enhance_bonus=0.03,
    level_cost_factor=157.0,
    enhance_base=800.0, enhance_growth=1.4696,
    success=np.array([95,93,88,82,77,71,66,60,55,49,44,38,33,27,22,16,11,9,5,3],float)/100,
    inv_expand_cost=5000.0, inv_max=30,
    refund=np.array([500,2000,8000,30000,100000],float),
    fuse_cost=np.array([1400,4300,14300,57300],float),
    drop=np.array([0.02608,0.01279,0.00237,0.000074,0.0],float),
    fuse_success_scale=0.902,
    storage_ratio=0.3, storage_stack_max=3,
    chosung_random=1000.0, chosung_category=500.0, wrong_min=50.0, wrong_max=110.0,
    combo_multiplier=1.2,
    hunmin_word=100.0, hunmin_mvp=1000.0,
    jamo_easy=800.0, jamo_normal=1200.0, jamo_hard=1800.0,
    bonus_long=300.0, bonus_jong=200.0, bonus_first=500.0, bonus_streak3=1000.0,
)

# v3.2 탐색 범위: late share 제약을 맞추기 위해 enhance_growth를 낮게 개방
PARAM_RANGES = {
    "reward_scale":        (0.90, 1.55),
    "chosung_scale":       (0.90, 1.45),
    "jamo_scale":          (0.80, 1.55),
    "hunmin_scale":        (0.80, 1.45),
    "combo_multiplier":    (1.10, 1.50),

    "enhance_base":        (250.0, 650.0),
    "enhance_growth":      (1.04, 1.24),
    "success_early":       (0.75, 1.25),
    "success_mid":         (0.75, 1.40),
    "success_late":        (1.20, 3.00),

    "level_factor":        (70.0, 180.0),

    "drop_D":              (0.008, 0.050),
    "drop_C":              (0.003, 0.025),
    "drop_B":              (0.0005, 0.008),
    "drop_A":              (0.00003, 0.0015),

    "synth_cost_scale":    (0.50, 1.60),
    "synth_success_scale": (0.50, 1.20),
    "dismantle_scale":     (0.70, 1.40),

    "storage_ratio":       (0.20, 0.50),
    "effect_scale":        (0.60, 1.25),
    "inv_expand_cost":     (3000.0, 12000.0),

    "activity_scale":      (0.80, 1.60),
    "spend_low":           (0.08, 0.28),
    "spend_mid":           (0.20, 0.55),
    "spend_high":          (0.40, 0.80),
    "spend_whale":         (0.35, 0.90),
    "fuse_aggr":           (0.05, 0.85),
    "dismantle_aggr":      (0.05, 0.50),
}

PARAM_NAMES = list(PARAM_RANGES.keys())
P_DIM = len(PARAM_NAMES)
print("P_DIM:", P_DIM)

def params_df_to_tensor(df):
    return torch.tensor(df[PARAM_NAMES].values, dtype=torch.float32, device=DEVICE)

def sample_sobol_params(n, seed=42):
    engine = torch.quasirandom.SobolEngine(dimension=P_DIM, scramble=True, seed=seed)
    u = engine.draw(n).numpy()
    out = {}
    for j, name in enumerate(PARAM_NAMES):
        lo, hi = PARAM_RANGES[name]
        out[name] = lo + (hi-lo)*u[:, j]
    return pd.DataFrame(out)

def sample_elite_refined(elite_df, n, shrink=0.20, seed=123):
    rng = np.random.default_rng(seed)
    elite = elite_df[PARAM_NAMES].values
    center_idx = rng.integers(0, len(elite), size=n)
    base = elite[center_idx].copy()

    out = base.copy()
    for j, name in enumerate(PARAM_NAMES):
        lo, hi = PARAM_RANGES[name]
        span = hi-lo
        noise = rng.normal(0, span*shrink, size=n)
        out[:, j] = np.clip(out[:, j] + noise, lo, hi)
    return pd.DataFrame(out, columns=PARAM_NAMES)

P_DIM: 28


In [6]:
# ================================================================
# 4. 강화 기대비용 계산 함수
# ================================================================

BASE_SUCCESS_T = torch.tensor(BASE["success"], dtype=torch.float32, device=DEVICE)

def make_success_rates(theta):
    """
    theta: [B, P]
    return success rates [B, 20]
    early: 0~7, mid: 8~14, late: 15~19
    """
    idx = {name:i for i,name in enumerate(PARAM_NAMES)}
    Bn = theta.shape[0]
    s = BASE_SUCCESS_T[None, :].repeat(Bn, 1)

    s[:, 0:8]  *= theta[:, idx["success_early"]][:,None]
    s[:, 8:15] *= theta[:, idx["success_mid"]][:,None]
    s[:, 15:20]*= theta[:, idx["success_late"]][:,None]
    s = torch.clamp(s, 0.03, 0.95)

    if ENFORCE_MONOTONIC_SUCCESS:
        # 강화 단계가 올라갈수록 성공률이 증가하지 않도록 보정
        s = torch.cummin(s, dim=1).values
    return s

def d20_expected_cost_and_late_share(theta):
    idx = {name:i for i,name in enumerate(PARAM_NAMES)}
    h = torch.arange(20, dtype=torch.float32, device=DEVICE)[None, :]
    enhance_base = theta[:, idx["enhance_base"]][:,None]
    enhance_growth = theta[:, idx["enhance_growth"]][:,None]
    succ = make_success_rates(theta)

    # D등급 기준 grade=1
    costs = torch.floor(enhance_base * (enhance_growth ** h))
    steps = costs / torch.clamp(succ, min=1e-6)
    total = steps.sum(dim=1)
    late = steps[:, 15:20].sum(dim=1) / torch.clamp(total, min=1.0)
    return total, late, steps, succ

# 기존형 baseline 확인
base_row = {
    "reward_scale":1.0,"chosung_scale":1.0,"jamo_scale":1.0,"hunmin_scale":1.0,"combo_multiplier":1.2,
    "enhance_base":800.0,"enhance_growth":1.4696,"success_early":1.0,"success_mid":1.0,"success_late":1.0,
    "level_factor":157.0,
    "drop_D":0.02608,"drop_C":0.01279,"drop_B":0.00237,"drop_A":0.000074,
    "synth_cost_scale":1.0,"synth_success_scale":0.902,"dismantle_scale":1.0,
    "storage_ratio":0.3,"effect_scale":1.0,"inv_expand_cost":5000.0,
    "activity_scale":1.0,"spend_low":0.12,"spend_mid":0.25,"spend_high":0.60,"spend_whale":0.70,
    "fuse_aggr":0.2,"dismantle_aggr":0.2
}
base_theta = params_df_to_tensor(pd.DataFrame([base_row]))
d20, late, steps, succ = d20_expected_cost_and_late_share(base_theta)
print("기존형 D20 EV:", float(d20[0].detach().cpu()), "late share:", float(late[0].detach().cpu()))
print("success:", succ[0].detach().cpu().numpy())

기존형 D20 EV: 69508568.0 late share: 0.9733669757843018
success: [0.95 0.93 0.88 0.82 0.77 0.71 0.66 0.6  0.55 0.49 0.44 0.38 0.33 0.27
 0.22 0.16 0.11 0.09 0.05 0.03]


In [7]:
# ================================================================
# 5. 유저 텐서 준비
# ================================================================

import numpy as np
import pandas as pd
import torch

# ------------------------------------------------
# 1) 필수 컬럼 보정
# ------------------------------------------------
if "accuracy_frac" not in users.columns:
    if "accuracy" in users.columns:
        acc = pd.to_numeric(users["accuracy"], errors="coerce").fillna(0.0).astype(float)
        users["accuracy_frac"] = np.where(acc > 1.0, acc / 100.0, acc)
    else:
        users["accuracy_frac"] = np.where(
            pd.to_numeric(users["total"], errors="coerce").fillna(0).values > 0,
            pd.to_numeric(users["correct"], errors="coerce").fillna(0).values
            / np.maximum(pd.to_numeric(users["total"], errors="coerce").fillna(0).values, 1),
            0.0,
        )

users["accuracy_frac"] = pd.to_numeric(users["accuracy_frac"], errors="coerce").fillna(0.0).clip(0.0, 1.0)

if "recent_active" not in users.columns:
    users["recent_active"] = users["last_attendance"].astype(str).str.contains(
        r"2026-05-0[3-5]|2026-05-04|2026-05-05", regex=True
    ).astype(float)

if "is_active" not in users.columns:
    users["is_active"] = pd.to_numeric(users["total"], errors="coerce").fillna(0) > 0

if "segment" not in users.columns:
    users["segment"] = "mid"
    users.loc[~users["is_active"], "segment"] = "inactive"

for col in ["score", "correct", "total", "recent_active"]:
    if col not in users.columns:
        users[col] = 0
    users[col] = pd.to_numeric(users[col], errors="coerce").fillna(0.0).astype(float)

users["segment"] = users["segment"].fillna("inactive").astype(str)

# ------------------------------------------------
# 2) 세그먼트별 행동 파라미터 및 segment id
# ------------------------------------------------
SEGMENT_SPEND_MAP = {
    "inactive": 0.03,
    "low": 0.12,
    "mid": 0.30,
    "high": 0.55,
    "whale": 0.70,
    "ultra": 0.75,
}
users["base_spend_prop"] = users["segment"].map(SEGMENT_SPEND_MAP).fillna(0.25).astype(float)

# simulate_batch()의 spend_table은 0~4 인덱스를 사용하므로 ultra는 whale 그룹(4)에 합친다.
SEGMENT_ID_MAP = {"inactive": 0, "low": 1, "mid": 2, "high": 3, "whale": 4, "ultra": 4}
users["segment_id"] = users["segment"].map(SEGMENT_ID_MAP).fillna(1).astype(int)

# ------------------------------------------------
# 3) 기본 daily activity 근사
# ------------------------------------------------
u_score_np = users["score"].to_numpy(dtype=np.float32)
u_correct_np = users["correct"].to_numpy(dtype=np.float32)
u_total_np = users["total"].to_numpy(dtype=np.float32)
u_acc_np = users["accuracy_frac"].to_numpy(dtype=np.float32)
u_recent_np = users["recent_active"].to_numpy(dtype=np.float32)
u_spend_np = users["base_spend_prop"].to_numpy(dtype=np.float32)
u_segment_np = users["segment_id"].to_numpy(dtype=np.int64)

base_daily_correct_np = (
    0.04
    + 0.035 * np.sqrt(np.clip(u_correct_np, 0, None))
    + 0.15 * u_recent_np
)
base_daily_correct_np = base_daily_correct_np * (0.55 + 0.90 * np.clip(u_acc_np, 0, 1))
base_daily_correct_np = np.clip(base_daily_correct_np, 0.01, 8.0).astype(np.float32)
users["base_daily_correct"] = base_daily_correct_np

# ------------------------------------------------
# 4) torch tensor 변환
# ------------------------------------------------
u_points = torch.tensor(u_score_np, dtype=torch.float32, device=DEVICE)
u_score = u_points  # 호환 alias
u_correct = torch.tensor(u_correct_np, dtype=torch.float32, device=DEVICE)
u_total = torch.tensor(u_total_np, dtype=torch.float32, device=DEVICE)
u_acc = torch.tensor(u_acc_np, dtype=torch.float32, device=DEVICE)
u_recent = torch.tensor(u_recent_np, dtype=torch.float32, device=DEVICE)
u_base_spend = torch.tensor(u_spend_np, dtype=torch.float32, device=DEVICE)
u_segment = torch.tensor(u_segment_np, dtype=torch.long, device=DEVICE)
BASE_DAILY_CORRECT = torch.tensor(base_daily_correct_np, dtype=torch.float32, device=DEVICE)

# ------------------------------------------------
# 5) 뒤쪽 함수 호환용 전역 별칭
# ------------------------------------------------
USER_POINTS = u_points
USER_SCORE = u_score
USER_CORRECT = u_correct
USER_TOTAL = u_total
USER_ACC = u_acc
USER_RECENT = u_recent
USER_BASE_SPEND = u_base_spend
USER_SEGMENT = u_segment

N_USERS = len(users)

print("N_USERS:", N_USERS)
print("device:", DEVICE)
print("score tensor:", u_points.shape, u_points.dtype)
print("accuracy_frac range:", float(u_acc.min().detach().cpu()), float(u_acc.max().detach().cpu()))
print("BASE_DAILY_CORRECT range:", float(BASE_DAILY_CORRECT.min().detach().cpu()), float(BASE_DAILY_CORRECT.max().detach().cpu()))
print("segment ids:", sorted(users["segment_id"].unique().tolist()))

display(
    users.groupby("segment")
    .agg(
        n=("nickname", "count"),
        mean_score=("score", "mean"),
        median_score=("score", "median"),
        mean_correct=("correct", "mean"),
        mean_accuracy_frac=("accuracy_frac", "mean"),
        mean_base_daily_correct=("base_daily_correct", "mean"),
        mean_spend_prop=("base_spend_prop", "mean"),
    )
    .reset_index()
)

users.to_csv(OUT_DIR / "parsed_users_segments_tensor_ready.csv", index=False, encoding="utf-8-sig")


N_USERS: 6945
device: cuda
score tensor: torch.Size([6945]) torch.float32
accuracy_frac range: 0.0 1.0
BASE_DAILY_CORRECT range: 0.02199999988079071 2.872703790664673
segment ids: [0, 1, 2, 3, 4]


,segment,n,mean_score,median_score,mean_correct,mean_accuracy_frac,mean_base_daily_correct,mean_spend_prop
0,high,489,3091.343558,1848.0,43.797546,0.352098,0.270245,0.55
1,inactive,692,46.764451,45.0,0.000000,0.000000,0.022000,0.03
2,low,2805,55.187879,36.0,0.262032,0.138391,0.036386,0.12
3,mid,2766,420.667751,279.5,4.772234,0.398849,0.100652,0.30
4,ultra,58,17896.672414,13735.5,261.482759,0.462642,0.722153,0.75
5,whale,135,4081.444444,3470.0,57.659259,0.366781,0.394893,0.70


In [8]:
# ================================================================
# 6. GPU 시뮬레이터
# ================================================================

def compute_game_reward(theta):
    """
    후보별 평균 정답 보상 추정.
    실제 게임을 모두 이벤트 단위로 돌리기보다 평균 보상으로 근사.
    """
    idx = {name:i for i,name in enumerate(PARAM_NAMES)}
    # 초성 평균: 랜덤 70%, 주제 30%, 오답차감 평균 반영
    chosung = (0.7*BASE["chosung_random"] + 0.3*BASE["chosung_category"] - 0.25*(BASE["wrong_min"]+BASE["wrong_max"])/2)
    chosung = chosung * theta[:, idx["chosung_scale"]]

    # 자모 평균: easy 35%, normal 45%, hard 20% + 평균 보너스
    jamo = (0.35*BASE["jamo_easy"] + 0.45*BASE["jamo_normal"] + 0.20*BASE["jamo_hard"]
            + 0.55*BASE["bonus_long"] + 0.45*BASE["bonus_jong"] + 0.20*BASE["bonus_first"] + 0.18*BASE["bonus_streak3"])
    jamo = jamo * theta[:, idx["jamo_scale"]]

    # 훈민 평균: 단어보상 + MVP 희석
    hunmin = (BASE["hunmin_word"]*1.8 + BASE["hunmin_mvp"]*0.06) * theta[:, idx["hunmin_scale"]]

    reward = 0.45*chosung + 0.40*jamo + 0.15*hunmin
    reward = reward * theta[:, idx["reward_scale"]]

    # 콤보 효과. 1.2를 기준으로 상대 변화만 부드럽게 반영
    combo_boost = 1.0 + 0.12*(theta[:, idx["combo_multiplier"]] - 1.0) / 0.2
    reward = reward * torch.clamp(combo_boost, 0.9, 1.4)
    return reward

@torch.no_grad()
def simulate_batch(theta, r_mc=4, t_days=180):
    idx = {name:i for i,name in enumerate(PARAM_NAMES)}
    Bn = theta.shape[0]
    R = r_mc
    N = N_USERS
    device = theta.device

    points = u_points[None,None,:].repeat(Bn,R,1)
    enhance = torch.zeros((Bn,R,N), dtype=torch.long, device=device)
    level = torch.ones((Bn,R,N), dtype=torch.long, device=device)
    main_grade = torch.ones((Bn,R,N), dtype=torch.long, device=device)

    relic_counts = torch.zeros((Bn,R,N,5), dtype=torch.float32, device=device)
    relic_counts[...,0] = 1.0

    earned_total = torch.zeros((Bn,R,N), dtype=torch.float32, device=device)
    spent_total = torch.zeros((Bn,R,N), dtype=torch.float32, device=device)

    avg_reward = compute_game_reward(theta)[:,None,None]
    activity_scale = theta[:, idx["activity_scale"]][:,None,None]

    base_daily = BASE_DAILY_CORRECT[None,None,:] * activity_scale
    seg = u_segment[None,None,:]

    spend_table = torch.stack([
        torch.zeros(Bn, device=device),
        theta[:, idx["spend_low"]],
        theta[:, idx["spend_mid"]],
        theta[:, idx["spend_high"]],
        theta[:, idx["spend_whale"]],
    ], dim=1)
    spend_ratio = spend_table[:,None,:].gather(2, seg.repeat(Bn,1,1)).clamp(0, 0.95)

    base_effect = torch.tensor(BASE["base_effect"], dtype=torch.float32, device=device)[None,None,None,:]
    base_effect = base_effect * theta[:, idx["effect_scale"]][:,None,None,None]

    succ = make_success_rates(theta)
    refund = torch.tensor(BASE["refund"], dtype=torch.float32, device=device)[None,None,None,:] * theta[:, idx["dismantle_scale"]][:,None,None,None]
    drop_probs = torch.stack([
        theta[:, idx["drop_D"]],
        theta[:, idx["drop_C"]],
        theta[:, idx["drop_B"]],
        theta[:, idx["drop_A"]],
        torch.zeros(Bn, device=device)
    ], dim=1).clamp(0, 0.20)

    grade_values = torch.arange(1,6,device=device).float()[None,None,None,:]

    for day in range(1, t_days+1):
        noise = torch.exp(0.35*torch.randn((Bn,R,N), device=device))
        lam = torch.clamp(base_daily * noise, min=0.001, max=30.0)
        daily_correct = torch.poisson(lam)

        gidx = torch.clamp(main_grade-1, 0, 4)
        effect_g = torch.gather(
            base_effect.repeat(1,R,N,1), 3, gidx[...,None]
        ).squeeze(-1)
        effect = effect_g * (1 + (level.float()-1)*BASE["level_bonus"]) * (1 + enhance.float()*BASE["enhance_bonus"])
        effect = torch.clamp(effect, 0, 1.0)

        reward = daily_correct * avg_reward * (1 + effect)
        points += reward
        earned_total += reward

        drops = []
        for gi in range(5):
            lam_drop = daily_correct * drop_probs[:,gi][:,None,None]
            drops.append(torch.poisson(torch.clamp(lam_drop, 0, 1000)))
        drops = torch.stack(drops, dim=-1)
        relic_counts += drops

        has_grade = relic_counts > 0
        max_grade = (has_grade.float()*grade_values).max(dim=-1).values.long().clamp(min=1)
        main_grade = torch.maximum(main_grade, max_grade)

        # 레벨업
        level_factor = theta[:, idx["level_factor"]][:,None,None]
        lvl_cost = main_grade.float() * level_factor * level.float()
        lvl_try = (level < 30) & (points * spend_ratio > lvl_cost) & (torch.rand_like(points) < 0.025)
        points -= torch.where(lvl_try, lvl_cost, torch.zeros_like(points))
        spent_total += torch.where(lvl_try, lvl_cost, torch.zeros_like(points))
        level = torch.where(lvl_try, level+1, level)

        # 강화
        for _ in range(SIM_MAX_ENHANCE_TRIES_PER_DAY):
            h = enhance.clamp(0,19)
            h_float = h.float()
            cost = torch.floor(main_grade.float() * theta[:,idx["enhance_base"]][:,None,None] *
                               (theta[:,idx["enhance_growth"]][:,None,None] ** h_float))
            can_try = (enhance < 20) & (points * spend_ratio > cost) & (torch.rand_like(points) < 0.35)
            p = torch.gather(succ[:,None,:].repeat(1,R,1), 2, h.reshape(Bn,R,N)).reshape(Bn,R,N)
            success = can_try & (torch.rand_like(points) < p)
            points -= torch.where(can_try, cost, torch.zeros_like(points))
            spent_total += torch.where(can_try, cost, torch.zeros_like(points))
            enhance = torch.where(success, enhance+1, enhance)

        # 합성/분해
        if day % FUSE_INTERVAL_DAYS == 0:
            fuse_aggr = theta[:,idx["fuse_aggr"]][:,None,None]
            synth_cost_scale = theta[:,idx["synth_cost_scale"]][:,None,None]
            synth_success_scale = theta[:,idx["synth_success_scale"]][:,None,None]
            fuse_cost_base = torch.tensor(BASE["fuse_cost"], dtype=torch.float32, device=device)

            for gi in range(4):
                possible_sets = torch.floor(relic_counts[...,gi] / 3.0)
                do_fuse = (possible_sets >= 1) & (torch.rand((Bn,R,N),device=device) < fuse_aggr*0.25)
                cost = fuse_cost_base[gi] * synth_cost_scale
                can_pay = points > cost
                attempt = do_fuse & can_pay
                base_prob = ((gi+1)*15 + 16) / 100.0
                prob = torch.clamp(base_prob * synth_success_scale, 0.05, 0.95)
                success_fuse = attempt & (torch.rand((Bn,R,N),device=device) < prob)
                relic_counts[...,gi] -= torch.where(attempt, torch.tensor(3.0,device=device), torch.tensor(0.0,device=device))
                relic_counts[...,gi+1] += success_fuse.float()
                points -= torch.where(attempt, cost, torch.zeros_like(points))
                spent_total += torch.where(attempt, cost, torch.zeros_like(points))

            dismantle_aggr = theta[:,idx["dismantle_aggr"]][:,None,None,None]
            excess = torch.clamp(relic_counts - 6.0, min=0.0)
            do_dis = (torch.rand_like(excess) < dismantle_aggr*0.10)
            dis_count = torch.where(do_dis, torch.floor(excess*0.5), torch.zeros_like(excess))
            gain = (dis_count * refund).sum(dim=-1)
            relic_counts -= dis_count
            points += gain
            earned_total += gain

            has_grade = relic_counts > 0
            max_grade = (has_grade.float()*grade_values).max(dim=-1).values.long().clamp(min=1)
            main_grade = torch.maximum(main_grade, max_grade)

    reach_d5 = (enhance >= 5).float().mean(dim=(1,2))
    reach_d10 = (enhance >= 10).float().mean(dim=(1,2))
    reach_d15 = (enhance >= 15).float().mean(dim=(1,2))
    reach_d20 = (enhance >= 20).float().mean(dim=(1,2))
    has_A = (relic_counts[...,3:].sum(dim=-1) > 0).float().mean(dim=(1,2))
    has_S = (relic_counts[...,4] > 0).float().mean(dim=(1,2))

    sink_ratio = spent_total.sum(dim=(1,2)) / torch.clamp(earned_total.sum(dim=(1,2)), min=1.0)

    p50 = torch.quantile(points.reshape(Bn,-1), 0.50, dim=1)
    p99 = torch.quantile(points.reshape(Bn,-1), 0.99, dim=1)
    p99_p50 = p99 / torch.clamp(p50, min=1.0)

    whale_mask = (u_segment == 4)[None,None,:]
    whale_spent = torch.where(whale_mask, spent_total, torch.zeros_like(spent_total)).sum(dim=(1,2))
    whale_earned = torch.where(whale_mask, earned_total, torch.zeros_like(earned_total)).sum(dim=(1,2))
    whale_sink_ratio = whale_spent / torch.clamp(whale_earned, min=1.0)

    d20_ev, late_share, _, _ = d20_expected_cost_and_late_share(theta)

    constraint_ok = (
        (d20_ev >= TARGET_D20_MIN) &
        (d20_ev <= TARGET_D20_MAX) &
        (late_share <= TARGET_LATE_SHARE_MAX) &
        (reach_d5 >= 0.25) & (reach_d5 <= 0.60) &
        (reach_d10 >= 0.05) & (reach_d10 <= 0.22) &
        (reach_d15 <= MAX_REACH_D15) &
        (reach_d20 <= MAX_REACH_D20) &
        (has_A <= MAX_HAS_A) &
        (has_S <= MAX_HAS_S) &
        (sink_ratio >= MIN_SINK) & (sink_ratio <= MAX_SINK) &
        (p99_p50 <= MAX_P99_P50)
    )

    loss = torch.zeros(Bn, device=device)
    loss += 1.5*torch.abs(reach_d5 - TARGET_REACH_D5)
    loss += 2.0*torch.abs(reach_d10 - TARGET_REACH_D10)
    loss += 3.0*torch.relu(reach_d15 - MAX_REACH_D15)
    loss += 5.0*torch.relu(reach_d20 - MAX_REACH_D20)
    loss += 3.0*torch.relu(has_A - MAX_HAS_A)
    loss += 10.0*torch.relu(has_S - MAX_HAS_S)
    loss += 2.0*torch.abs(sink_ratio - TARGET_SINK)
    loss += 1.5*torch.relu(p99_p50 - MAX_P99_P50)/MAX_P99_P50
    loss += 5.0*torch.relu(d20_ev - TARGET_D20_MAX)/TARGET_D20_MAX
    loss += 2.0*torch.relu(TARGET_D20_MIN - d20_ev)/TARGET_D20_MIN
    loss += 8.0*torch.relu(late_share - TARGET_LATE_SHARE_MAX)

    return {
        "balance_loss": loss.detach().cpu().numpy(),
        "constraint_ok": constraint_ok.detach().cpu().numpy().astype(bool),
        "reach_d5": reach_d5.detach().cpu().numpy(),
        "reach_d10": reach_d10.detach().cpu().numpy(),
        "reach_d15": reach_d15.detach().cpu().numpy(),
        "reach_d20": reach_d20.detach().cpu().numpy(),
        "has_A": has_A.detach().cpu().numpy(),
        "has_S": has_S.detach().cpu().numpy(),
        "sink_ratio": sink_ratio.detach().cpu().numpy(),
        "p99_p50": p99_p50.detach().cpu().numpy(),
        "D20_ev_cost": d20_ev.detach().cpu().numpy(),
        "late_15_20_share": late_share.detach().cpu().numpy(),
        "whale_sink_ratio": whale_sink_ratio.detach().cpu().numpy(),
    }

In [9]:
# ================================================================
# 7. 대량 탐색 실행 함수
# ================================================================

def run_sweep(param_df, r_mc, batch_size, stage_name):
    all_parts = []
    n = len(param_df)
    print(f"[{stage_name}] n={n}, r_mc={r_mc}, batch={batch_size}")

    for start in range(0, n, batch_size):
        end = min(start+batch_size, n)
        batch_df = param_df.iloc[start:end].copy()
        theta = params_df_to_tensor(batch_df)

        metrics = simulate_batch(theta, r_mc=r_mc, t_days=T_DAYS)
        mdf = pd.DataFrame(metrics)
        out = pd.concat([batch_df.reset_index(drop=True), mdf.reset_index(drop=True)], axis=1)
        out["stage"] = stage_name
        out["theta_id"] = np.arange(start, end)

        all_parts.append(out)

        if SAVE_EVERY_BATCH:
            tmp = pd.concat(all_parts, ignore_index=True)
            tmp = tmp.sort_values(["constraint_ok","balance_loss"], ascending=[False, True]).head(KEEP_TOP_K)
            tmp.to_csv(OUT_DIR/f"{stage_name}_running_top.csv", index=False, encoding="utf-8-sig")

        if (start//batch_size) % 10 == 0:
            gc.collect()
            if DEVICE=="cuda":
                torch.cuda.empty_cache()
            ok_count = sum(part["constraint_ok"].sum() for part in all_parts)
            print(f"  {end}/{n} done | constraint_ok so far: {ok_count}")

    res = pd.concat(all_parts, ignore_index=True)
    res = res.sort_values(["constraint_ok","balance_loss"], ascending=[False, True]).reset_index(drop=True)
    res.to_csv(OUT_DIR/f"{stage_name}_all_results.csv", index=False, encoding="utf-8-sig")
    res.head(KEEP_TOP_K).to_csv(OUT_DIR/f"{stage_name}_top{KEEP_TOP_K}.csv", index=False, encoding="utf-8-sig")
    print(f"[{stage_name}] done. constraint_ok={res['constraint_ok'].sum()} / {len(res)}")
    display(res.head(10))
    return res

In [10]:
# ================================================================
# 8. 1차 대량 탐색
# ================================================================

stage1_params = sample_sobol_params(STAGE1_N_PARAM, seed=SEED)
stage1_params.to_csv(OUT_DIR/"stage1_sampled_params.csv", index=False, encoding="utf-8-sig")

stage1_res = run_sweep(
    stage1_params,
    r_mc=STAGE1_R_MC,
    batch_size=STAGE1_BATCH_SIZE,
    stage_name="stage1_qmc_wide"
)

print("stage1 constraint_ok:", stage1_res["constraint_ok"].sum())

[stage1_qmc_wide] n=65536, r_mc=4, batch=1024
  1024/65536 done | constraint_ok so far: 0
  11264/65536 done | constraint_ok so far: 0
  21504/65536 done | constraint_ok so far: 0
  31744/65536 done | constraint_ok so far: 0
  41984/65536 done | constraint_ok so far: 0
  52224/65536 done | constraint_ok so far: 0
  62464/65536 done | constraint_ok so far: 0
[stage1_qmc_wide] done. constraint_ok=0 / 65536


,reward_scale,chosung_scale,jamo_scale,hunmin_scale,combo_multiplier,enhance_base,enhance_growth,success_early,success_mid,success_late,...,reach_d20,has_A,has_S,sink_ratio,p99_p50,D20_ev_cost,late_15_20_share,whale_sink_ratio,stage,theta_id
0,1.037183,0.937618,0.947246,1.102876,1.275672,629.194458,1.120072,1.167902,0.768097,2.771466,...,0.000360,0.003852,0.0,0.766315,3.996656,240984.015625,0.701459,0.971574,stage1_qmc_wide,48650
1,0.909437,0.954684,1.242691,0.906548,1.134325,620.507202,1.122982,0.875326,0.806287,2.722239,...,0.000756,0.001548,0.0,0.795259,3.763316,245739.921875,0.704094,0.974190,stage1_qmc_wide,1122
2,0.914806,1.105379,0.873090,1.231078,1.213107,630.669434,1.177421,0.823125,0.778392,2.945369,...,0.000000,0.010043,0.0,0.768846,4.101678,521615.312500,0.751514,0.976185,stage1_qmc_wide,10146
3,1.062632,1.118773,1.014706,0.860717,1.169919,633.418213,1.117245,0.789447,0.863742,2.150776,...,0.000828,0.021130,0.0,0.777441,3.610975,250841.296875,0.730261,1.006569,stage1_qmc_wide,10102
4,0.917867,1.320188,0.898707,1.396123,1.279102,554.107483,1.146801,0.824482,0.761124,2.682099,...,0.000324,0.021778,0.0,0.779919,3.957279,317518.687500,0.727467,0.981448,stage1_qmc_wide,3293
5,1.182505,0.973928,1.097814,1.271738,1.338510,629.152710,1.132383,0.757214,0.900500,2.983399,...,0.000864,0.002664,0.0,0.805076,4.333672,259963.593750,0.708273,0.963654,stage1_qmc_wide,27022
6,0.936029,1.015363,0.810785,0.840130,1.295392,532.856384,1.143768,0.994519,0.813281,2.820146,...,0.000180,0.022750,0.0,0.739995,3.455560,274712.187500,0.729265,0.999930,stage1_qmc_wide,11266
7,1.026714,0.949154,0.930446,1.201753,1.296182,576.271362,1.154538,0.849846,0.855627,2.660880,...,0.000180,0.010691,0.0,0.733912,3.574357,347707.375000,0.748926,0.982266,stage1_qmc_wide,15050
8,0.964204,1.085292,0.889868,1.352031,1.406964,633.425415,1.129751,1.129797,0.792343,2.366431,...,0.000324,0.017459,0.0,0.715137,3.272298,293634.593750,0.735121,1.009148,stage1_qmc_wide,61682
9,0.947272,1.119132,1.165484,1.435582,1.237875,453.062744,1.155091,0.778832,0.771532,2.715774,...,0.000180,0.009755,0.0,0.716043,3.421704,288077.875000,0.735007,1.013700,stage1_qmc_wide,850


stage1 constraint_ok: 0


In [11]:
# ================================================================
# 9. 2차 elite 주변 재탐색
# ================================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------
# 0) stage1_res 복구
# ------------------------------------------------
stage1_file_candidates = [
    OUT_DIR / "stage1_qmc_wide_all_results.csv",
    OUT_DIR / "stage1_qmc_wide_results.csv",
    OUT_DIR / "stage1_results.csv",
]

if "stage1_res" not in globals():
    loaded = False
    for fp in stage1_file_candidates:
        if fp.exists():
            print(f"stage1_res 변수가 없어 CSV에서 복구합니다: {fp}")
            stage1_res = pd.read_csv(fp)
            loaded = True
            break
    if not loaded:
        raise NameError(
            "stage1_res가 없고, stage1 결과 CSV도 찾지 못했습니다. "
            "먼저 8번 '1차 대량탐색' 섹션을 정상적으로 실행해주세요."
        )
else:
    print("메모리의 stage1_res를 사용합니다.")

stage1_res = stage1_res.copy()

if "balance_loss" not in stage1_res.columns:
    raise KeyError("stage1_res에 balance_loss 컬럼이 없습니다. 8번 결과 생성 여부를 확인하세요.")

if "constraint_ok" not in stage1_res.columns:
    print("constraint_ok 컬럼이 없어 False로 생성합니다.")
    stage1_res["constraint_ok"] = False

if stage1_res["constraint_ok"].dtype == object:
    stage1_res["constraint_ok"] = stage1_res["constraint_ok"].astype(str).str.lower().isin(["true", "1", "yes"])

stage1_res = stage1_res.sort_values("balance_loss", ascending=True).reset_index(drop=True)
print("stage1 result shape:", stage1_res.shape)
print("constraint_ok count:", int(stage1_res["constraint_ok"].sum()))
print("best stage1 loss:", float(stage1_res["balance_loss"].iloc[0]))

# ------------------------------------------------
# 1) elite pool 선택
# ------------------------------------------------
ELITE_POOL_SIZE = min(2000, len(stage1_res))

if stage1_res["constraint_ok"].sum() > 0:
    elite_pool = stage1_res[stage1_res["constraint_ok"]].sort_values("balance_loss").head(ELITE_POOL_SIZE).copy()
    print(f"constraint_ok=True 후보에서 elite_pool 구성: {len(elite_pool)}개")
else:
    elite_pool = stage1_res.sort_values("balance_loss").head(ELITE_POOL_SIZE).copy()
    print(f"constraint_ok=True 후보가 없어 balance_loss 상위 후보로 elite_pool 구성: {len(elite_pool)}개")

if len(elite_pool) < 5:
    print(f"주의: elite_pool이 매우 작습니다: {len(elite_pool)}개. 실제 본실험에서는 8번 탐색 결과가 충분히 커야 합니다.")

elite_pool.to_csv(OUT_DIR / "stage2_elite_pool.csv", index=False, encoding="utf-8-sig")
display(elite_pool.head(20))

# ------------------------------------------------
# 2) 2차 탐색 후보 생성
# ------------------------------------------------
stage2_params = sample_elite_refined(elite_pool, STAGE2_N_PARAM, shrink=0.10, seed=SEED + 1)
stage2_params.to_csv(OUT_DIR / "stage2_refined_sampled_params.csv", index=False, encoding="utf-8-sig")

print("stage2_params shape:", stage2_params.shape)
display(stage2_params.head())

# ------------------------------------------------
# 3) 2차 재탐색 실행
# ------------------------------------------------
print(f"[stage2_elite_refined] n={len(stage2_params)}, r_mc={STAGE2_R_MC}, batch={STAGE2_BATCH_SIZE}")

stage2_res = run_sweep(
    stage2_params,
    r_mc=STAGE2_R_MC,
    batch_size=STAGE2_BATCH_SIZE,
    stage_name="stage2_elite_refined",
)

stage2_res = stage2_res.sort_values(["constraint_ok", "balance_loss"], ascending=[False, True]).reset_index(drop=True)
stage2_res.to_csv(OUT_DIR / "stage2_elite_refined_all_results.csv", index=False, encoding="utf-8-sig")

print("stage2 result shape:", stage2_res.shape)
print("constraint_ok count:", int(stage2_res["constraint_ok"].sum()))
print("best stage2 loss:", float(stage2_res["balance_loss"].iloc[0]))
display(stage2_res.head(30))


메모리의 stage1_res를 사용합니다.
stage1 result shape: (65536, 43)
constraint_ok count: 0
best stage1 loss: 0.5486590266227722
constraint_ok=True 후보가 없어 balance_loss 상위 후보로 elite_pool 구성: 2000개


,reward_scale,chosung_scale,jamo_scale,hunmin_scale,combo_multiplier,enhance_base,enhance_growth,success_early,success_mid,success_late,...,reach_d20,has_A,has_S,sink_ratio,p99_p50,D20_ev_cost,late_15_20_share,whale_sink_ratio,stage,theta_id
0,1.037183,0.937618,0.947246,1.102876,1.275672,629.194458,1.120072,1.167902,0.768097,2.771466,...,0.000360,0.003852,0.0,0.766315,3.996656,240984.015625,0.701459,0.971574,stage1_qmc_wide,48650
1,0.909437,0.954684,1.242691,0.906548,1.134325,620.507202,1.122982,0.875326,0.806287,2.722239,...,0.000756,0.001548,0.0,0.795259,3.763316,245739.921875,0.704094,0.974190,stage1_qmc_wide,1122
2,0.914806,1.105379,0.873090,1.231078,1.213107,630.669434,1.177421,0.823125,0.778392,2.945369,...,0.000000,0.010043,0.0,0.768846,4.101678,521615.312500,0.751514,0.976185,stage1_qmc_wide,10146
3,1.062632,1.118773,1.014706,0.860717,1.169919,633.418213,1.117245,0.789447,0.863742,2.150776,...,0.000828,0.021130,0.0,0.777441,3.610975,250841.296875,0.730261,1.006569,stage1_qmc_wide,10102
4,0.917867,1.320188,0.898707,1.396123,1.279102,554.107483,1.146801,0.824482,0.761124,2.682099,...,0.000324,0.021778,0.0,0.779919,3.957279,317518.687500,0.727467,0.981448,stage1_qmc_wide,3293
5,1.182505,0.973928,1.097814,1.271738,1.338510,629.152710,1.132383,0.757214,0.900500,2.983399,...,0.000864,0.002664,0.0,0.805076,4.333672,259963.593750,0.708273,0.963654,stage1_qmc_wide,27022
6,0.936029,1.015363,0.810785,0.840130,1.295392,532.856384,1.143768,0.994519,0.813281,2.820146,...,0.000180,0.022750,0.0,0.739995,3.455560,274712.187500,0.729265,0.999930,stage1_qmc_wide,11266
7,1.026714,0.949154,0.930446,1.201753,1.296182,576.271362,1.154538,0.849846,0.855627,2.660880,...,0.000180,0.010691,0.0,0.733912,3.574357,347707.375000,0.748926,0.982266,stage1_qmc_wide,15050
8,0.964204,1.085292,0.889868,1.352031,1.406964,633.425415,1.129751,1.129797,0.792343,2.366431,...,0.000324,0.017459,0.0,0.715137,3.272298,293634.593750,0.735121,1.009148,stage1_qmc_wide,61682
9,0.947272,1.119132,1.165484,1.435582,1.237875,453.062744,1.155091,0.778832,0.771532,2.715774,...,0.000180,0.009755,0.0,0.716043,3.421704,288077.875000,0.735007,1.013700,stage1_qmc_wide,850


stage2_params shape: (131072, 28)


,reward_scale,chosung_scale,jamo_scale,hunmin_scale,combo_multiplier,enhance_base,enhance_growth,success_early,success_mid,success_late,...,storage_ratio,effect_scale,inv_expand_cost,activity_scale,spend_low,spend_mid,spend_high,spend_whale,fuse_aggr,dismantle_aggr
0,0.904226,1.315891,0.821900,1.340707,1.212330,650.000000,1.156088,1.231580,1.007490,1.865770,...,0.352343,1.250000,8910.842773,0.807171,0.184802,0.250268,0.558568,0.482817,0.129847,0.091987
1,1.068930,0.947865,0.835321,1.141074,1.340951,410.358093,1.146085,0.914747,1.029938,2.923001,...,0.239039,0.851472,10546.874023,1.514628,0.141797,0.257688,0.488277,0.550166,0.235637,0.374254
2,0.959074,1.107073,0.981491,1.047495,1.107214,650.000000,1.224271,0.919285,0.780895,2.521935,...,0.290634,0.934546,8234.285156,1.567231,0.177409,0.376286,0.768001,0.350000,0.118302,0.304615
3,1.143911,1.248966,0.909208,1.098256,1.109014,610.564331,1.122903,0.929997,0.986087,2.576234,...,0.439443,0.967703,11185.006836,0.883983,0.195812,0.344221,0.797015,0.884403,0.050000,0.445591
4,0.900000,1.160710,1.383925,1.051567,1.150188,650.000000,1.083689,0.967590,0.750000,2.238448,...,0.287847,0.687123,9538.439453,1.339875,0.146695,0.340490,0.603291,0.723570,0.654397,0.450275


[stage2_elite_refined] n=131072, r_mc=4, batch=1024
[stage2_elite_refined] n=131072, r_mc=4, batch=1024
  1024/131072 done | constraint_ok so far: 0
  11264/131072 done | constraint_ok so far: 1
  21504/131072 done | constraint_ok so far: 2
  31744/131072 done | constraint_ok so far: 4
  41984/131072 done | constraint_ok so far: 4
  52224/131072 done | constraint_ok so far: 5
  62464/131072 done | constraint_ok so far: 6
  72704/131072 done | constraint_ok so far: 6
  82944/131072 done | constraint_ok so far: 6
  93184/131072 done | constraint_ok so far: 6
  103424/131072 done | constraint_ok so far: 8
  113664/131072 done | constraint_ok so far: 10
  123904/131072 done | constraint_ok so far: 10
[stage2_elite_refined] done. constraint_ok=10 / 131072


,reward_scale,chosung_scale,jamo_scale,hunmin_scale,combo_multiplier,enhance_base,enhance_growth,success_early,success_mid,success_late,...,reach_d20,has_A,has_S,sink_ratio,p99_p50,D20_ev_cost,late_15_20_share,whale_sink_ratio,stage,theta_id
0,0.900000,0.962708,1.059811,1.337300,1.215837,536.937378,1.136569,0.770720,0.750000,3.000000,...,0.000252,0.009539,0.000000,0.763458,4.444549,257626.718750,0.698194,0.969516,stage2_elite_refined,112439
1,0.937156,1.006273,1.247584,0.932094,1.171063,650.000000,1.123199,0.791789,0.784630,3.000000,...,0.000432,0.005616,0.000000,0.769671,3.950306,252962.046875,0.686040,0.985433,stage2_elite_refined,94934
2,0.988005,0.922391,0.946008,1.190613,1.326604,650.000000,1.123458,1.219953,0.750000,3.000000,...,0.000144,0.002952,0.000000,0.746851,3.683075,255821.078125,0.694309,0.973243,stage2_elite_refined,109402
3,0.959922,1.079379,0.869041,1.176183,1.330047,642.860168,1.123729,0.991454,0.750000,3.000000,...,0.000324,0.018790,0.000000,0.714768,3.375505,255609.875000,0.690161,1.020152,stage2_elite_refined,55583
4,1.489820,1.237122,0.800028,1.356042,1.120389,650.000000,1.126114,0.750000,0.754841,2.849905,...,0.000648,0.008207,0.000000,0.745878,4.145518,275305.937500,0.690855,0.916428,stage2_elite_refined,31104
5,1.140214,1.008040,1.075300,0.936095,1.133271,633.282776,1.125432,0.750000,0.760765,2.651923,...,0.000432,0.018790,0.000000,0.796133,5.124196,272593.218750,0.699761,0.931717,stage2_elite_refined,28341
6,1.021995,1.419414,0.957443,1.149624,1.141526,650.000000,1.119831,0.817482,0.750000,2.721892,...,0.000576,0.014003,0.000000,0.781694,3.820618,256948.406250,0.691021,0.958394,stage2_elite_refined,12598
7,0.951880,0.921324,0.823587,1.299407,1.315572,595.133484,1.131485,0.765546,0.750000,2.958497,...,0.000612,0.017747,0.000000,0.807997,3.325339,267948.000000,0.693300,1.018385,stage2_elite_refined,10705
8,1.128424,0.900000,1.223869,0.800000,1.192077,613.126526,1.125901,0.966985,0.763667,2.962402,...,0.000576,0.016955,0.000072,0.738511,3.924842,250220.953125,0.695320,0.955009,stage2_elite_refined,44175
9,1.089271,0.900000,0.800000,1.237159,1.316243,650.000000,1.126862,0.903518,0.781653,2.982328,...,0.000360,0.017171,0.000000,0.835766,4.281451,265462.593750,0.695743,1.008530,stage2_elite_refined,103301


stage2 result shape: (131072, 43)
constraint_ok count: 10
best stage2 loss: 0.22332711517810822


,reward_scale,chosung_scale,jamo_scale,hunmin_scale,combo_multiplier,enhance_base,enhance_growth,success_early,success_mid,success_late,...,reach_d20,has_A,has_S,sink_ratio,p99_p50,D20_ev_cost,late_15_20_share,whale_sink_ratio,stage,theta_id
0,0.900000,0.962708,1.059811,1.337300,1.215837,536.937378,1.136569,0.770720,0.750000,3.000000,...,0.000252,0.009539,0.000000,0.763458,4.444549,257626.718750,0.698194,0.969516,stage2_elite_refined,112439
1,0.937156,1.006273,1.247584,0.932094,1.171063,650.000000,1.123199,0.791789,0.784630,3.000000,...,0.000432,0.005616,0.000000,0.769671,3.950306,252962.046875,0.686040,0.985433,stage2_elite_refined,94934
2,0.988005,0.922391,0.946008,1.190613,1.326604,650.000000,1.123458,1.219953,0.750000,3.000000,...,0.000144,0.002952,0.000000,0.746851,3.683075,255821.078125,0.694309,0.973243,stage2_elite_refined,109402
3,0.959922,1.079379,0.869041,1.176183,1.330047,642.860168,1.123729,0.991454,0.750000,3.000000,...,0.000324,0.018790,0.000000,0.714768,3.375505,255609.875000,0.690161,1.020152,stage2_elite_refined,55583
4,1.489820,1.237122,0.800028,1.356042,1.120389,650.000000,1.126114,0.750000,0.754841,2.849905,...,0.000648,0.008207,0.000000,0.745878,4.145518,275305.937500,0.690855,0.916428,stage2_elite_refined,31104
5,1.140214,1.008040,1.075300,0.936095,1.133271,633.282776,1.125432,0.750000,0.760765,2.651923,...,0.000432,0.018790,0.000000,0.796133,5.124196,272593.218750,0.699761,0.931717,stage2_elite_refined,28341
6,1.021995,1.419414,0.957443,1.149624,1.141526,650.000000,1.119831,0.817482,0.750000,2.721892,...,0.000576,0.014003,0.000000,0.781694,3.820618,256948.406250,0.691021,0.958394,stage2_elite_refined,12598
7,0.951880,0.921324,0.823587,1.299407,1.315572,595.133484,1.131485,0.765546,0.750000,2.958497,...,0.000612,0.017747,0.000000,0.807997,3.325339,267948.000000,0.693300,1.018385,stage2_elite_refined,10705
8,1.128424,0.900000,1.223869,0.800000,1.192077,613.126526,1.125901,0.966985,0.763667,2.962402,...,0.000576,0.016955,0.000072,0.738511,3.924842,250220.953125,0.695320,0.955009,stage2_elite_refined,44175
9,1.089271,0.900000,0.800000,1.237159,1.316243,650.000000,1.126862,0.903518,0.781653,2.982328,...,0.000360,0.017171,0.000000,0.835766,4.281451,265462.593750,0.695743,1.008530,stage2_elite_refined,103301


In [12]:
# ================================================================
# 10. 최종 정밀검증
# ================================================================

import pandas as pd

# stage1/stage2 결과가 메모리에 없으면 CSV에서 복구
if "stage1_res" not in globals():
    fp = OUT_DIR / "stage1_qmc_wide_all_results.csv"
    if not fp.exists():
        raise NameError("stage1_res가 없고 stage1_qmc_wide_all_results.csv도 없습니다. 8번 섹션을 먼저 실행하세요.")
    stage1_res = pd.read_csv(fp)

if "stage2_res" not in globals():
    fp = OUT_DIR / "stage2_elite_refined_all_results.csv"
    if not fp.exists():
        raise NameError("stage2_res가 없고 stage2_elite_refined_all_results.csv도 없습니다. 9번 섹션을 먼저 실행하세요.")
    stage2_res = pd.read_csv(fp)

for df_name in ["stage1_res", "stage2_res"]:
    df = globals()[df_name]
    if "constraint_ok" in df.columns and df["constraint_ok"].dtype == object:
        df["constraint_ok"] = df["constraint_ok"].astype(str).str.lower().isin(["true", "1", "yes"])
    globals()[df_name] = df

combined = pd.concat([stage1_res, stage2_res], ignore_index=True)
combined = combined.sort_values(["constraint_ok", "balance_loss"], ascending=[False, True]).reset_index(drop=True)
combined.to_csv(OUT_DIR / "combined_stage1_stage2_ranked.csv", index=False, encoding="utf-8-sig")

validate_df = combined.head(VALIDATE_TOP_N)[PARAM_NAMES].copy()
validate_df.to_csv(OUT_DIR / "validation_input_top_candidates.csv", index=False, encoding="utf-8-sig")

validation_res = run_sweep(
    validate_df,
    r_mc=VALIDATE_R_MC,
    batch_size=min(128, max(16, VALIDATE_TOP_N)),
    stage_name="stage3_deep_validation",
)

validation_res.to_csv(OUT_DIR / "FINAL_deep_validation_results.csv", index=False, encoding="utf-8-sig")
display(validation_res.head(20))


[stage3_deep_validation] n=500, r_mc=64, batch=128
  128/500 done | constraint_ok so far: 10
[stage3_deep_validation] done. constraint_ok=10 / 500


,reward_scale,chosung_scale,jamo_scale,hunmin_scale,combo_multiplier,enhance_base,enhance_growth,success_early,success_mid,success_late,...,reach_d20,has_A,has_S,sink_ratio,p99_p50,D20_ev_cost,late_15_20_share,whale_sink_ratio,stage,theta_id
0,0.900000,0.962708,1.059811,1.337300,1.215837,536.937378,1.136569,0.770720,0.750000,3.000000,...,0.000234,0.009029,0.000000,0.760746,4.422726,257626.718750,0.698194,0.965010,stage3_deep_validation,0
1,0.988005,0.922391,0.946008,1.190613,1.326604,650.000000,1.123458,1.219953,0.750000,3.000000,...,0.000263,0.003604,0.000000,0.745357,3.764981,255821.078125,0.694309,0.971124,stage3_deep_validation,2
2,0.937156,1.006273,1.247584,0.932094,1.171063,650.000000,1.123199,0.791789,0.784630,3.000000,...,0.000517,0.005193,0.000000,0.768994,3.925304,252962.046875,0.686040,0.986074,stage3_deep_validation,1
3,0.959922,1.079379,0.869041,1.176183,1.330047,642.860168,1.123729,0.991454,0.750000,3.000000,...,0.000398,0.019047,0.000002,0.714270,3.397068,255609.875000,0.690161,1.019312,stage3_deep_validation,3
4,1.489820,1.237122,0.800028,1.356042,1.120389,650.000000,1.126114,0.750000,0.754841,2.849905,...,0.000637,0.008234,0.000000,0.747086,4.053299,275305.937500,0.690855,0.919672,stage3_deep_validation,4
5,1.140214,1.008040,1.075300,0.936095,1.133271,633.282776,1.125432,0.750000,0.760765,2.651923,...,0.000328,0.019598,0.000000,0.797357,5.175458,272593.218750,0.699761,0.929532,stage3_deep_validation,5
6,1.021995,1.419414,0.957443,1.149624,1.141526,650.000000,1.119831,0.817482,0.750000,2.721892,...,0.000430,0.015254,0.000000,0.781597,3.834897,256948.406250,0.691021,0.956581,stage3_deep_validation,6
7,0.951880,0.921324,0.823587,1.299407,1.315572,595.133484,1.131485,0.765546,0.750000,2.958497,...,0.000648,0.017670,0.000000,0.809594,3.326219,267948.000000,0.693300,1.023645,stage3_deep_validation,7
8,1.128424,0.900000,1.223869,0.800000,1.192077,613.126526,1.125901,0.966985,0.763667,2.962402,...,0.000614,0.018032,0.000000,0.738805,3.954723,250220.953125,0.695320,0.951360,stage3_deep_validation,8
9,1.089271,0.900000,0.800000,1.237159,1.316243,650.000000,1.126862,0.903518,0.781653,2.982328,...,0.000391,0.016894,0.000002,0.834241,4.208797,265462.593750,0.695743,1.004578,stage3_deep_validation,9


,reward_scale,chosung_scale,jamo_scale,hunmin_scale,combo_multiplier,enhance_base,enhance_growth,success_early,success_mid,success_late,...,reach_d20,has_A,has_S,sink_ratio,p99_p50,D20_ev_cost,late_15_20_share,whale_sink_ratio,stage,theta_id
0,0.900000,0.962708,1.059811,1.337300,1.215837,536.937378,1.136569,0.770720,0.750000,3.000000,...,0.000234,0.009029,0.000000,0.760746,4.422726,257626.718750,0.698194,0.965010,stage3_deep_validation,0
1,0.988005,0.922391,0.946008,1.190613,1.326604,650.000000,1.123458,1.219953,0.750000,3.000000,...,0.000263,0.003604,0.000000,0.745357,3.764981,255821.078125,0.694309,0.971124,stage3_deep_validation,2
2,0.937156,1.006273,1.247584,0.932094,1.171063,650.000000,1.123199,0.791789,0.784630,3.000000,...,0.000517,0.005193,0.000000,0.768994,3.925304,252962.046875,0.686040,0.986074,stage3_deep_validation,1
3,0.959922,1.079379,0.869041,1.176183,1.330047,642.860168,1.123729,0.991454,0.750000,3.000000,...,0.000398,0.019047,0.000002,0.714270,3.397068,255609.875000,0.690161,1.019312,stage3_deep_validation,3
4,1.489820,1.237122,0.800028,1.356042,1.120389,650.000000,1.126114,0.750000,0.754841,2.849905,...,0.000637,0.008234,0.000000,0.747086,4.053299,275305.937500,0.690855,0.919672,stage3_deep_validation,4
5,1.140214,1.008040,1.075300,0.936095,1.133271,633.282776,1.125432,0.750000,0.760765,2.651923,...,0.000328,0.019598,0.000000,0.797357,5.175458,272593.218750,0.699761,0.929532,stage3_deep_validation,5
6,1.021995,1.419414,0.957443,1.149624,1.141526,650.000000,1.119831,0.817482,0.750000,2.721892,...,0.000430,0.015254,0.000000,0.781597,3.834897,256948.406250,0.691021,0.956581,stage3_deep_validation,6
7,0.951880,0.921324,0.823587,1.299407,1.315572,595.133484,1.131485,0.765546,0.750000,2.958497,...,0.000648,0.017670,0.000000,0.809594,3.326219,267948.000000,0.693300,1.023645,stage3_deep_validation,7
8,1.128424,0.900000,1.223869,0.800000,1.192077,613.126526,1.125901,0.966985,0.763667,2.962402,...,0.000614,0.018032,0.000000,0.738805,3.954723,250220.953125,0.695320,0.951360,stage3_deep_validation,8
9,1.089271,0.900000,0.800000,1.237159,1.316243,650.000000,1.126862,0.903518,0.781653,2.982328,...,0.000391,0.016894,0.000002,0.834241,4.208797,265462.593750,0.695743,1.004578,stage3_deep_validation,9


In [13]:
# ================================================================
# 11. 최종 후보 추출 및 사람이 보기 좋은 파라미터 정리
# ================================================================

final = validation_res.sort_values(["constraint_ok","balance_loss"], ascending=[False, True]).reset_index(drop=True)
best = final.iloc[0].to_dict()

def make_success_array_from_row(row):
    df = pd.DataFrame([{name: row[name] for name in PARAM_NAMES}])
    theta = params_df_to_tensor(df)
    s = make_success_rates(theta)[0].detach().cpu().numpy()
    return np.round(s*100, 2)

best_success = make_success_array_from_row(best)

def rint(x, step=1):
    return int(round(float(x)/step)*step)

recommended = {
    "theta_id": int(best.get("theta_id", final.index[0])),
    "constraint_ok": bool(best["constraint_ok"]),
    "metrics": {k: float(best[k]) for k in [
        "balance_loss","reach_d5","reach_d10","reach_d15","reach_d20",
        "has_A","has_S","sink_ratio","p99_p50","D20_ev_cost",
        "late_15_20_share","whale_sink_ratio"
    ]},
    "raw_params": {name: float(best[name]) for name in PARAM_NAMES},
    "code_params": {
        "chosung_random": rint(BASE["chosung_random"]*best["reward_scale"]*best["chosung_scale"], 10),
        "chosung_category": rint(BASE["chosung_category"]*best["reward_scale"]*best["chosung_scale"], 10),
        "wrong_min": rint(BASE["wrong_min"]*best["reward_scale"]*best["chosung_scale"], 10),
        "wrong_max": rint(BASE["wrong_max"]*best["reward_scale"]*best["chosung_scale"], 10),
        "hunmin_word": rint(BASE["hunmin_word"]*best["reward_scale"]*best["hunmin_scale"], 10),
        "hunmin_mvp": rint(BASE["hunmin_mvp"]*best["reward_scale"]*best["hunmin_scale"], 10),
        "jamo_easy": rint(BASE["jamo_easy"]*best["reward_scale"]*best["jamo_scale"], 10),
        "jamo_normal": rint(BASE["jamo_normal"]*best["reward_scale"]*best["jamo_scale"], 10),
        "jamo_hard": rint(BASE["jamo_hard"]*best["reward_scale"]*best["jamo_scale"], 10),
        "bonus_long": rint(BASE["bonus_long"]*best["reward_scale"]*best["jamo_scale"], 10),
        "bonus_jong": rint(BASE["bonus_jong"]*best["reward_scale"]*best["jamo_scale"], 10),
        "bonus_first": rint(BASE["bonus_first"]*best["reward_scale"]*best["jamo_scale"], 10),
        "bonus_streak3": rint(BASE["bonus_streak3"]*best["reward_scale"]*best["jamo_scale"], 10),
        "combo_multiplier": round(float(best["combo_multiplier"]), 2),

        "base_effect_percent": np.round(BASE["base_effect"]*best["effect_scale"]*100, 2).tolist(),
        "level_bonus_per_level": BASE["level_bonus"],
        "enhance_bonus_per_enhance": BASE["enhance_bonus"],

        "level_cost_factor": rint(best["level_factor"], 1),
        "enhance_cost_base": rint(best["enhance_base"], 1),
        "enhance_cost_growth": round(float(best["enhance_growth"]), 4),
        "enhance_success_rate_percent": best_success.tolist(),

        "drop_rate": {
            "D": round(float(best["drop_D"]), 6),
            "C": round(float(best["drop_C"]), 6),
            "B": round(float(best["drop_B"]), 6),
            "A": round(float(best["drop_A"]), 6),
            "S": 0.0,
        },

        "fuse_cost": np.round(BASE["fuse_cost"]*best["synth_cost_scale"]).astype(int).tolist(),
        "fuse_success_scale": round(float(best["synth_success_scale"]), 4),
        "dismantle_refund": np.round(BASE["refund"]*best["dismantle_scale"]).astype(int).tolist(),

        "storage_effect_ratio": round(float(best["storage_ratio"]), 3),
        "storage_stack_max": 3,
        "inventory_expand_cost": rint(best["inv_expand_cost"], 10),
        "inventory_expand_max": 30,
    }
}

with open(OUT_DIR/"FINAL_recommended_parameters_v32.json", "w", encoding="utf-8") as f:
    json.dump(recommended, f, ensure_ascii=False, indent=2)

pd.DataFrame([recommended["code_params"]]).to_csv(OUT_DIR/"FINAL_code_params_v32_flat.csv", index=False, encoding="utf-8-sig")

print(json.dumps(recommended, ensure_ascii=False, indent=2)[:4000])

{
  "theta_id": 0,
  "constraint_ok": true,
  "metrics": {
    "balance_loss": 0.21419896185398102,
    "reach_d5": 0.4868452847003937,
    "reach_d10": 0.15121939778327942,
    "reach_d15": 0.010965622961521149,
    "reach_d20": 0.00023398127814289182,
    "has_A": 0.00902852788567543,
    "has_S": 0.0,
    "sink_ratio": 0.7607461214065552,
    "p99_p50": 4.422726154327393,
    "D20_ev_cost": 257626.71875,
    "late_15_20_share": 0.6981944441795349,
    "whale_sink_ratio": 0.9650096297264099
  },
  "raw_params": {
    "reward_scale": 0.8999999761581421,
    "chosung_scale": 0.9627083539962769,
    "jamo_scale": 1.0598106384277344,
    "hunmin_scale": 1.3372998237609863,
    "combo_multiplier": 1.2158374786376953,
    "enhance_base": 536.9373779296875,
    "enhance_growth": 1.1365690231323242,
    "success_early": 0.7707196474075317,
    "success_mid": 0.75,
    "success_late": 3.0,
    "level_factor": 179.12982177734375,
    "drop_D": 0.029327193275094032,
    "drop_C": 0.022768611088

In [14]:
# ================================================================
# 12. 개발자용 message 생성
# ================================================================

cp = recommended["code_params"]
m = recommended["metrics"]

message = f"""
초능력자 봇 유물/게임경제 파라미터 v3.2 대량탐색 적용 후보입니다.

[선정 기준]
- 후보 수: stage1 {STAGE1_N_PARAM:,}개 + stage2 {STAGE2_N_PARAM:,}개
- 최종 검증: 상위 {VALIDATE_TOP_N:,}개 후보 × MC {VALIDATE_R_MC}회
- 주요 제약:
  - D20 기대비용: {TARGET_D20_MIN:,} ~ {TARGET_D20_MAX:,}P
  - 15~20강 비용 집중도: {TARGET_LATE_SHARE_MAX:.2f} 이하
  - D+10 도달률: 5~22%
  - D+15 도달률: {MAX_REACH_D15:.1%} 이하
  - D+20 도달률: {MAX_REACH_D20:.1%} 이하
  - A 이상 보유율: {MAX_HAS_A:.1%} 이하
  - S 보유율: {MAX_HAS_S:.1%} 이하
  - sink ratio: {MIN_SINK:.2f}~{MAX_SINK:.2f}
  - P99/P50: {MAX_P99_P50:.1f} 이하

[최종 후보 지표]
- constraint_ok: {recommended['constraint_ok']}
- balance_loss: {m['balance_loss']:.4f}
- D+5 도달률: {m['reach_d5']:.3f}
- D+10 도달률: {m['reach_d10']:.3f}
- D+15 도달률: {m['reach_d15']:.3f}
- D+20 도달률: {m['reach_d20']:.5f}
- A 이상 보유율: {m['has_A']:.3f}
- S 보유율: {m['has_S']:.5f}
- sink ratio: {m['sink_ratio']:.3f}
- P99/P50: {m['p99_p50']:.2f}
- D20 기대비용: {m['D20_ev_cost']:.0f}P
- 15~20강 비용 집중도: {m['late_15_20_share']:.3f}
- whale sink ratio: {m['whale_sink_ratio']:.3f}

[게임 보상]
- 초성퀴즈 랜덤 모드 시작 포인트: {cp['chosung_random']}P
- 초성퀴즈 주제선택 시작 포인트: {cp['chosung_category']}P
- 오답/힌트 차감: {cp['wrong_min']}~{cp['wrong_max']}P
- 훈민정음 단어 보상: {cp['hunmin_word']}P
- 훈민정음 MVP 보너스: {cp['hunmin_mvp']}P
- 자모연성 easy/normal/hard: {cp['jamo_easy']}P / {cp['jamo_normal']}P / {cp['jamo_hard']}P
- 자모연성 긴 단어/받침/하루첫/3연속 보너스: {cp['bonus_long']}P / {cp['bonus_jong']}P / {cp['bonus_first']}P / {cp['bonus_streak3']}P
- 콤보 배율: x{cp['combo_multiplier']}

[유물 효과]
- 등급별 기본 효과(%): D/C/B/A/S = {cp['base_effect_percent']}
- 레벨당 효과 증가: +{cp['level_bonus_per_level']*100:.0f}%
- 강화당 효과 증가: +{cp['enhance_bonus_per_enhance']*100:.0f}%

[레벨업]
- MAX_LEVEL: 30
- LEVEL_UP_COST_FACTOR: {cp['level_cost_factor']}
- 공식: grade × {cp['level_cost_factor']} × currentLevel

[강화]
- MAX_ENHANCE: 20
- ENHANCE_COST_BASE: {cp['enhance_cost_base']}
- ENHANCE_COST_GROWTH: {cp['enhance_cost_growth']}
- 공식: floor(grade × {cp['enhance_cost_base']} × {cp['enhance_cost_growth']}^currentEnhance)
- 실패 시 강화 수치 유지, 비용만 소모
- 성공률(%): {cp['enhance_success_rate_percent']}

[드랍률]
- D: {cp['drop_rate']['D']:.4%}
- C: {cp['drop_rate']['C']:.4%}
- B: {cp['drop_rate']['B']:.4%}
- A: {cp['drop_rate']['A']:.4%}
- S: 0%

[합성]
- 비용 D→C/C→B/B→A/A→S: {cp['fuse_cost']}
- 합성 성공률: 기존 공식 결과 × {cp['fuse_success_scale']}
- clamp 5~95% 유지

[분해 환급]
- D/C/B/A/S: {cp['dismantle_refund']}

[보관함]
- 보관 효과율: {cp['storage_effect_ratio']:.1%}
- 같은 효과 타입 최대 중첩: {cp['storage_stack_max']}개
- 슬롯 확장비: {cp['inventory_expand_cost']}P
- 최대 슬롯: {cp['inventory_expand_max']}

[주의]
이 후보는 시뮬레이션 기반 추천안입니다. 실제 배포 후 2주 동안 강화 시도, 합성 시도, 드랍량, A/S 보유자 수, 포인트 P99/P50, sink ratio를 매일 기록한 뒤 재조정해야 합니다.
""".strip()

Path(OUT_DIR/"developer_message_v32.txt").write_text(message, encoding="utf-8")
print(message[:5000])

초능력자 봇 유물/게임경제 파라미터 v3.2 대량탐색 적용 후보입니다.

[선정 기준]
- 후보 수: stage1 65,536개 + stage2 131,072개
- 최종 검증: 상위 500개 후보 × MC 64회
- 주요 제약:
  - D20 기대비용: 250,000 ~ 2,000,000P
  - 15~20강 비용 집중도: 0.70 이하
  - D+10 도달률: 5~22%
  - D+15 도달률: 3.0% 이하
  - D+20 도달률: 0.1% 이하
  - A 이상 보유율: 2.0% 이하
  - S 보유율: 0.1% 이하
  - sink ratio: 0.65~0.90
  - P99/P50: 7.0 이하

[최종 후보 지표]
- constraint_ok: True
- balance_loss: 0.2142
- D+5 도달률: 0.487
- D+10 도달률: 0.151
- D+15 도달률: 0.011
- D+20 도달률: 0.00023
- A 이상 보유율: 0.009
- S 보유율: 0.00000
- sink ratio: 0.761
- P99/P50: 4.42
- D20 기대비용: 257627P
- 15~20강 비용 집중도: 0.698
- whale sink ratio: 0.965

[게임 보상]
- 초성퀴즈 랜덤 모드 시작 포인트: 870P
- 초성퀴즈 주제선택 시작 포인트: 430P
- 오답/힌트 차감: 40~100P
- 훈민정음 단어 보상: 120P
- 훈민정음 MVP 보너스: 1200P
- 자모연성 easy/normal/hard: 760P / 1140P / 1720P
- 자모연성 긴 단어/받침/하루첫/3연속 보너스: 290P / 190P / 480P / 950P
- 콤보 배율: x1.22

[유물 효과]
- 등급별 기본 효과(%): D/C/B/A/S = [2.37, 4.73, 9.36, 17.64, 29.36]
- 레벨당 효과 증가: +5%
- 강화당 효과 증가: +3%

[레벨업]
- MAX_LEVEL: 30
- LEVEL_UP_COST_FACTOR: 17

## 실행 후 확인할 파일

- `stage1_qmc_wide_all_results.csv`
- `stage2_elite_refined_all_results.csv`
- `FINAL_deep_validation_results.csv`
- `FINAL_recommended_parameters_v32.json`
- `FINAL_code_params_v32_flat.csv`
- `developer_message_v32.txt`

중요 체크:
1. `constraint_ok=True` 후보가 몇 개 나왔는지
2. 최종 후보의 `late_15_20_share`가 0.70 이하인지
3. 최종 후보의 `D20_ev_cost`가 250,000~2,000,000P 사이인지
4. `reach_d10`이 너무 낮거나 높지 않은지
5. `has_A`, `has_S`, `p99_p50`이 과도하지 않은지

만약 여전히 `constraint_ok=True` 후보가 없으면, 단일 지수식 강화비용 공식으로는 목표를 동시에 만족시키기 어렵다는 뜻입니다. 그 경우 다음 단계는 `grade × base × growth^enhance`를 버리고, 0~20강 비용 테이블을 직접 최적화하는 방식으로 넘어가는 것이 좋습니다.

In [16]:
# ================================================================
# 결과 전체 ZIP 다운로드
# ================================================================

from pathlib import Path
import shutil
from google.colab import files
import datetime

# 결과 저장 폴더
OUT_DIR = Path("/content/chaoneung_v32_outputs")

# 혹시 다른 이름으로 저장했을 경우 후보 폴더 자동 탐색
if not OUT_DIR.exists():
    candidates = [
        Path("/content/outputs"),
        Path("/content/artifact_balance_outputs"),
        Path("/content/chaoneung_param_lab_v32_outputs"),
        Path("/content/chaoneung_param_lab_outputs"),
    ]
    for c in candidates:
        if c.exists():
            OUT_DIR = c
            break

if not OUT_DIR.exists():
    raise FileNotFoundError("결과 폴더를 찾지 못했습니다. OUT_DIR 경로를 확인해주세요.")

timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")
zip_base = Path(f"/content/chaoneung_results_{timestamp}")
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(OUT_DIR))

print("ZIP 생성 완료:", zip_path)
print("포함된 파일 수:", len(list(OUT_DIR.rglob("*"))))

files.download(zip_path)

ZIP 생성 완료: /content/chaoneung_results_20260504_205952.zip
포함된 파일 수: 20


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>